In [2]:
!pip install pingouin factor_analyzer -q

In [3]:
!pip install --upgrade numpy scipy pingouin factor_analyzer

In [6]:
# %% [Langkah 1: Persiapan Lingkungan dan Impor Pustaka]
import os
import re
import pandas as pd
import numpy as np
from scipy import stats

try:
    from IPython.display import display
except ImportError:
    display = print

# Mengatur tampilan tabel agar lebar kolom sama persis seperti di Google Colab
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_columns", 10)
pd.set_option("display.width", 1000)


# %% [Langkah 2: Lokasi Berkas Data]
folder_path = r"D:\Probstat"
if not os.path.exists(folder_path):
    folder_path = "."

nama_file = os.path.join(folder_path, "Blank Quiz (Responses).xlsx")
print("File terbaca:", nama_file)


# %% [Langkah 3: Membaca Data dan Memisahkan Butir]
# Menggunakan 37 responden pertama (.iloc[:37], sebelum responden ke-38 pukul 16:16 masuk)
# serta memfilter kolom pertanyaan yang berformat "Dimensi [Pernyataan]"
JUMLAH_KOLOM_DEMOGRAFI = 6

df = pd.read_excel(nama_file).iloc[:37]
butir = df.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]
butir = butir.loc[:, butir.columns.str.contains(r"\[.*\]")]

print("Jumlah responden :", len(df))
print("Jumlah butir     :", butir.shape[1])
print("Data kosong      :", int(butir.isna().sum().sum()))


# %% [Langkah 4: Mengubah Jawaban Likert Menjadi Angka]
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}

# Melakukan replace dan mengonversi tipe data secara eksplisit untuk menghindari peringatan downcasting
num = butir.replace(skala).astype(float).apply(pd.to_numeric, errors="coerce")

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
display(num.head())


# %% [Langkah 5: Mengelompokkan Butir per Dimensi]
dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?) \[", kolom).group(1)
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")


# %% [Langkah 6: Uji Validitas (Korelasi Pearson Butir-Total)]
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
    x = num[daftar]
    total = x.sum(axis=1)
    for i, kolom in enumerate(daftar, 1):
        r, p = stats.pearsonr(x[kolom], total)
        r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
        hasil_validitas.append({
            "Dimensi": nama,
            "Butir": f"B{i}",
            "r hitung": round(r, 3),
            "Corrected r": round(r_koreksi, 3),
            "p-value": round(p, 4),
            "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
        })

tabel_validitas = pd.DataFrame(hasil_validitas)
display(tabel_validitas)


# %% [Langkah 7: Uji Reliabilitas (Cronbach's Alpha)]
def cronbach_alpha(x):
    k = x.shape[1]
    return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
    return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
            else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []

for nama, daftar in dimensi.items():
    x = num[daftar]
    a = cronbach_alpha(x)
    hasil_reliabilitas.append({
        "Dimensi": nama,
        "Jumlah butir": len(daftar),
        "Cronbach's alpha": round(a, 3),
        "Kategori": kategori(a),
    })
    for i, kolom in enumerate(daftar, 1):
        if len(daftar) > 2:
            alpha_dihapus.append({
                "Dimensi": nama,
                "Butir": f"B{i}",
                "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3),
            })

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)


# %% [Langkah 8: Menggabungkan dan Menyimpan Hasil ke Excel]
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

nama_output = os.path.join(folder_path, "hasil_uji_validitas_reliabilitas.xlsx")
with pd.ExcelWriter(nama_output) as w:
    gabung.to_excel(w, sheet_name="Validitas", index=False)
    tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

print("Hasil berhasil disimpan ke:", nama_output)

File terbaca: ./Blank Quiz (Responses).xlsx
Jumlah responden : 37
Jumlah butir     : 29
Data kosong      : 0
Sel yang gagal dikonversi: 0


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],...,Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,3.0,4.0,4.0,4.0,3.0,...,4.0,3.0,4.0,4.0,3.0
1,4.0,4.0,4.0,2.0,3.0,...,4.0,4.0,4.0,3.0,4.0
2,3.0,4.0,5.0,3.0,4.0,...,4.0,4.0,3.0,4.0,4.0
3,5.0,5.0,5.0,5.0,4.0,...,4.0,5.0,4.0,5.0,5.0
4,5.0,4.0,4.0,3.0,4.0,...,3.0,4.0,5.0,4.0,5.0


Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
 Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik. : 4 butir
Rsspon emosional: 5 butir
n = 37 | r tabel (α=0,05; df=35) = 0.325


,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.793,0.701,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.732,0.624,0.0000,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.790,0.703,0.0000,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.433,0.295,0.0075,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.764,0.650,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.838,0.755,0.0000,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.746,0.627,0.0000,Valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.862,0.801,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.790,0.712,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.779,0.682,0.0000,Valid


,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.858,Baik
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.904,Sangat baik
2,Kebiasaan penggunaan teknologi digital,6,0.777,Reliabel
3,Prokrastinasi akademik.,4,0.846,Baik
4,Rsspon emosional,5,0.875,Baik


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.827
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.838
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.827
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.875
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.834
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.818
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.838
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.879
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.890
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.894


Hasil berhasil disimpan ke: ./hasil_uji_validitas_reliabilitas.xlsx


In [7]:
from google.colab import files

# Mengunduh berkas hasil analisis ke perangkat lokal
files.download('hasil_uji_validitas_reliabilitas.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>